# 🚀 BERT Fine-tuning for Fake News Detection

This notebook trains a BERT model to detect fake news using Google Colab's free GPU.

**Features:**
- Full BERT training with GPU acceleration
- Automatic model saving to Google Drive
- Real-time training metrics
- Model evaluation and testing


## 🔧 Setup and Installation

In [ ]:
# Install required packages
!pip install torch transformers scikit-learn pandas numpy matplotlib seaborn
!pip install accelerate datasets

# Mount Google Drive for data and model storage
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup,
    TrainingArguments,
    Trainer
)
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import os
from datetime import datetime
import json

# Set style for plots
plt.style.use('default')
sns.set_palette("husl")

print("✅ All imports successful!")
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"💻 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🚀 GPU: {torch.cuda.get_device_name(0)}")

## 📊 Data Upload and Preparation

Upload your `news.csv` file to Google Drive or use the file upload below.

In [ ]:
# Option 1: Upload file directly
from google.colab import files
print("📤 Upload your news.csv file:")
uploaded = files.upload()

# Get the uploaded filename
data_file = list(uploaded.keys())[0]
print(f"✅ Uploaded: {data_file}")

In [ ]:
# Option 2: Load from Google Drive (uncomment if using Drive)
# data_file = "/content/drive/MyDrive/news.csv"

# Load and explore the data
print("📊 Loading data...")
df = pd.read_csv(data_file)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print("\n📈 Label distribution:")
print(df['label'].value_counts())

# Display sample data
print("\n🔍 Sample data:")
df.head()

In [ ]:
# Visualize data distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Label distribution
df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['skyblue', 'lightcoral'])
axes[0].set_title('Label Distribution')
axes[0].set_xlabel('Label (0=Real, 1=Fake)')
axes[0].set_ylabel('Count')

# Text length distribution
df['text_length'] = df['text'].str.len()
df['text_length'].hist(bins=50, ax=axes[1], alpha=0.7, color='lightgreen')
axes[1].set_title('Text Length Distribution')
axes[1].set_xlabel('Characters')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

print(f"📏 Average text length: {df['text_length'].mean():.0f} characters")
print(f"📏 Median text length: {df['text_length'].median():.0f} characters")

## ⚙️ Configuration

In [ ]:
# Training Configuration
CONFIG = {
    'MODEL_NAME': 'bert-base-uncased',
    'MAX_LENGTH': 512,  # Full BERT context length
    'BATCH_SIZE': 16,   # Adjust based on GPU memory
    'EPOCHS': 3,
    'LEARNING_RATE': 2e-5,
    'WARMUP_STEPS': 500,
    'WEIGHT_DECAY': 0.01,
    'TEST_SIZE': 0.2,
    'VAL_SIZE': 0.1,
    'RANDOM_STATE': 42,
    'SAVE_STEPS': 500,
    'EVAL_STEPS': 500,
    'LOGGING_STEPS': 100
}

# Create output directory
OUTPUT_DIR = '/content/bert_fake_news_model'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("🔧 Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

# Save config
with open(f'{OUTPUT_DIR}/config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)

print(f"\n📁 Output directory: {OUTPUT_DIR}")

## 🏗️ Dataset Preparation

In [ ]:
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            return_attention_mask=True,
            return_tensors='pt',
            truncation=True
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

def create_data_loader(df, tokenizer, max_len, batch_size, shuffle=True):
    ds = NewsDataset(
        texts=df.text.to_numpy(),
        labels=df.label.to_numpy(),
        tokenizer=tokenizer,
        max_len=max_len
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

print("✅ Dataset class created!")

In [ ]:
# Split the data
print("🔀 Splitting data...")

# First split: train + val, test
df_train_val, df_test = train_test_split(
    df, 
    test_size=CONFIG['TEST_SIZE'], 
    random_state=CONFIG['RANDOM_STATE'],
    stratify=df['label']
)

# Second split: train, val
df_train, df_val = train_test_split(
    df_train_val,
    test_size=CONFIG['VAL_SIZE']/(1-CONFIG['TEST_SIZE']),
    random_state=CONFIG['RANDOM_STATE'],
    stratify=df_train_val['label']
)

print(f"📊 Data split:")
print(f"  Training: {len(df_train)} samples")
print(f"  Validation: {len(df_val)} samples")
print(f"  Test: {len(df_test)} samples")

# Verify label distribution
print("\n🎯 Label distribution:")
for split_name, split_df in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    dist = split_df['label'].value_counts(normalize=True)
    print(f"  {split_name}: Real={dist[0]:.3f}, Fake={dist[1]:.3f}")

In [ ]:
# Initialize tokenizer and create data loaders
print("🔤 Loading tokenizer...")
tokenizer = BertTokenizer.from_pretrained(CONFIG['MODEL_NAME'])

print("📦 Creating data loaders...")
train_data_loader = create_data_loader(
    df_train, tokenizer, CONFIG['MAX_LENGTH'], CONFIG['BATCH_SIZE']
)
val_data_loader = create_data_loader(
    df_val, tokenizer, CONFIG['MAX_LENGTH'], CONFIG['BATCH_SIZE']
)
test_data_loader = create_data_loader(
    df_test, tokenizer, CONFIG['MAX_LENGTH'], CONFIG['BATCH_SIZE'], shuffle=False
)

print(f"✅ Data loaders created!")
print(f"  Train batches: {len(train_data_loader)}")
print(f"  Val batches: {len(val_data_loader)}")
print(f"  Test batches: {len(test_data_loader)}")

## 🤖 Model Setup

In [ ]:
# Initialize model
print("🤖 Loading BERT model...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"💻 Using device: {device}")

model = BertForSequenceClassification.from_pretrained(
    CONFIG['MODEL_NAME'],
    num_labels=2,
    output_attentions=False,
    output_hidden_states=False
)

model = model.to(device)

# Setup optimizer and scheduler
optimizer = AdamW(
    model.parameters(),
    lr=CONFIG['LEARNING_RATE'],
    weight_decay=CONFIG['WEIGHT_DECAY']
)

total_steps = len(train_data_loader) * CONFIG['EPOCHS']
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=CONFIG['WARMUP_STEPS'],
    num_training_steps=total_steps
)

print(f"✅ Model setup complete!")
print(f"📊 Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"🔧 Total training steps: {total_steps:,}")

## 🏋️‍♀️ Training

In [ ]:
# Training and evaluation functions
def train_epoch(model, data_loader, optimizer, device, scheduler):
    model = model.train()
    losses = []
    correct_predictions = 0
    total_predictions = 0
    
    for batch_idx, d in enumerate(data_loader):
        input_ids = d["input_ids"].to(device)
        attention_mask = d["attention_mask"].to(device)
        labels = d["labels"].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        loss = outputs.loss
        preds = torch.argmax(outputs.logits, dim=1)
        correct_predictions += torch.sum(preds == labels)
        total_predictions += labels.size(0)
        losses.append(loss.item())
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        
        if batch_idx % CONFIG['LOGGING_STEPS'] == 0:
            print(f"    Batch {batch_idx}/{len(data_loader)}, Loss: {loss.item():.4f}")
    
    return correct_predictions.double() / total_predictions, np.mean(losses)

def eval_model(model, data_loader, device):
    model = model.eval()
    losses = []
    correct_predictions = 0
    total_predictions = 0
    
    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            labels = d["labels"].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            
            loss = outputs.loss
            preds = torch.argmax(outputs.logits, dim=1)
            correct_predictions += torch.sum(preds == labels)
            total_predictions += labels.size(0)
            losses.append(loss.item())
    
    return correct_predictions.double() / total_predictions, np.mean(losses)

print("✅ Training functions ready!")

In [ ]:
# Training loop with progress tracking
print("🚀 Starting training...")
print(f"⚙️ Configuration: {CONFIG['EPOCHS']} epochs, batch size {CONFIG['BATCH_SIZE']}")

# Training history
history = {
    'train_acc': [], 'val_acc': [],
    'train_loss': [], 'val_loss': []
}

best_val_acc = 0
start_time = datetime.now()

for epoch in range(CONFIG['EPOCHS']):
    print(f"\n🏃‍♂️ Epoch {epoch + 1}/{CONFIG['EPOCHS']}")
    print("-" * 60)
    
    epoch_start = datetime.now()
    
    # Train
    train_acc, train_loss = train_epoch(
        model, train_data_loader, optimizer, device, scheduler
    )
    
    # Validate
    val_acc, val_loss = eval_model(model, val_data_loader, device)
    
    # Update history
    history['train_acc'].append(train_acc.cpu().numpy())
    history['val_acc'].append(val_acc.cpu().numpy())
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    
    epoch_time = datetime.now() - epoch_start
    
    print(f"📊 Results:")
    print(f"    Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}")
    print(f"    Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")
    print(f"⏱️ Epoch time: {epoch_time}")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        print(f"💾 New best model! Saving...")
        model.save_pretrained(f"{OUTPUT_DIR}/best_model")
        tokenizer.save_pretrained(f"{OUTPUT_DIR}/best_model")

total_time = datetime.now() - start_time
print(f"\n🎉 Training completed!")
print(f"⏱️ Total time: {total_time}")
print(f"🏆 Best validation accuracy: {best_val_acc:.4f}")

# Save final model
model.save_pretrained(f"{OUTPUT_DIR}/final_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_model")

# Save training history
import pickle
with open(f'{OUTPUT_DIR}/training_history.pkl', 'wb') as f:
    pickle.dump(history, f)

print(f"💾 Models saved to {OUTPUT_DIR}")

## 📊 Training Visualization

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy plot
axes[0].plot(range(1, len(history['train_acc']) + 1), history['train_acc'], 'bo-', label='Training')
axes[0].plot(range(1, len(history['val_acc']) + 1), history['val_acc'], 'ro-', label='Validation')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss plot
axes[1].plot(range(1, len(history['train_loss']) + 1), history['train_loss'], 'bo-', label='Training')
axes[1].plot(range(1, len(history['val_loss']) + 1), history['val_loss'], 'ro-', label='Validation')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_history.png', dpi=300, bbox_inches='tight')
plt.show()

print("📈 Training plots saved!")

## 🧪 Model Evaluation

In [ ]:
# Load best model for evaluation
print("📥 Loading best model for evaluation...")
best_model = BertForSequenceClassification.from_pretrained(f"{OUTPUT_DIR}/best_model")
best_model = best_model.to(device)

# Evaluate on test set
print("🧪 Evaluating on test set...")
test_acc, test_loss = eval_model(best_model, test_data_loader, device)

print(f"🎯 Test Results:")
print(f"    Accuracy: {test_acc:.4f}")
print(f"    Loss: {test_loss:.4f}")

# Get detailed predictions for classification report
def get_predictions(model, data_loader, device):
    model = model.eval()
    predictions = []
    actual_labels = []
    
    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            labels = d["labels"].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)
            
            predictions.extend(preds.cpu().tolist())
            actual_labels.extend(labels.cpu().tolist())
    
    return actual_labels, predictions

y_true, y_pred = get_predictions(best_model, test_data_loader, device)

# Classification report
print("\n📋 Detailed Classification Report:")
print(classification_report(y_true, y_pred, target_names=['Real News', 'Fake News']))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Real', 'Fake'], 
            yticklabels=['Real', 'Fake'])
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig(f'{OUTPUT_DIR}/confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Evaluation complete!")

## 🎯 Test Your Model

In [ ]:
# Interactive testing function
def predict_news(text, model, tokenizer, device, max_length=512):
    model.eval()
    
    encoding = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=max_length,
        return_token_type_ids=False,
        padding='max_length',
        return_attention_mask=True,
        return_tensors='pt',
        truncation=True
    )
    
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        prediction = torch.nn.functional.softmax(outputs.logits, dim=-1)
        confidence = torch.max(prediction, dim=-1)
        
    predicted_class = torch.argmax(outputs.logits, dim=-1).cpu().numpy()[0]
    confidence_score = confidence.values.cpu().numpy()[0]
    
    return {
        'prediction': 'Fake News' if predicted_class == 1 else 'Real News',
        'confidence': confidence_score,
        'probabilities': {
            'Real News': prediction[0][0].cpu().numpy(),
            'Fake News': prediction[0][1].cpu().numpy()
        }
    }

# Test with sample texts
test_texts = [
    "Scientists have discovered a new planet that could support human life in a nearby solar system.",
    "BREAKING: Aliens have landed in New York City and are demanding to speak to our leader!",
    "The stock market experienced significant volatility today following the Federal Reserve's announcement.",
    "You won't believe this ONE WEIRD TRICK that doctors HATE! Click now to lose 50 pounds instantly!"
]

print("🎯 Testing the model with sample texts:\n")

for i, text in enumerate(test_texts, 1):
    result = predict_news(text, best_model, tokenizer, device)
    print(f"📰 Test {i}:")
    print(f"Text: {text[:100]}...")
    print(f"Prediction: {result['prediction']}")
    print(f"Confidence: {result['confidence']:.4f}")
    print(f"Probabilities: Real={result['probabilities']['Real News']:.4f}, Fake={result['probabilities']['Fake News']:.4f}")
    print("-" * 80)

print("✅ Sample testing complete!")

In [ ]:
# Interactive testing - input your own text
print("🔍 Test your own news article!")
print("Enter a news article text below:")

# Get user input
user_text = input("Your news text: ")

if user_text.strip():
    result = predict_news(user_text, best_model, tokenizer, device)
    
    print("\n" + "="*80)
    print("🔮 PREDICTION RESULTS")
    print("="*80)
    print(f"📰 Text: {user_text[:200]}..." if len(user_text) > 200 else f"📰 Text: {user_text}")
    print(f"\n🎯 Prediction: **{result['prediction']}**")
    print(f"📊 Confidence: {result['confidence']:.2%}")
    print(f"\n📈 Detailed Probabilities:")
    print(f"   • Real News: {result['probabilities']['Real News']:.2%}")
    print(f"   • Fake News: {result['probabilities']['Fake News']:.2%}")
    
    # Add interpretation
    if result['confidence'] > 0.8:
        print(f"\n💡 Interpretation: High confidence prediction")
    elif result['confidence'] > 0.6:
        print(f"\n💡 Interpretation: Moderate confidence prediction")
    else:
        print(f"\n💡 Interpretation: Low confidence - the model is uncertain")
    
    print("="*80)
else:
    print("❌ No text provided!")

## 💾 Download Your Models

Download the trained models to use locally or deploy.

In [ ]:
# Create a zip file with all models and results
import shutil
import zipfile
from google.colab import files

print("📦 Creating download package...")

# Create zip file
zip_path = '/content/bert_fake_news_models.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Add model files
    for root, dirs, files_list in os.walk(OUTPUT_DIR):
        for file in files_list:
            file_path = os.path.join(root, file)
            arc_path = os.path.relpath(file_path, '/content')
            zipf.write(file_path, arc_path)

print(f"✅ Package created: {zip_path}")
print(f"📁 Package size: {os.path.getsize(zip_path) / (1024*1024):.1f} MB")

# Download the package
print("⬇️ Downloading package...")
files.download(zip_path)

print("🎉 Download complete! Your trained BERT model is ready to use!")

## 📋 Training Summary

### What You've Accomplished:

1. **✅ Data Preparation**: Loaded and processed your fake news dataset
2. **✅ Model Training**: Fine-tuned BERT for fake news detection
3. **✅ Evaluation**: Achieved high accuracy on test data
4. **✅ Visualization**: Generated training plots and confusion matrix
5. **✅ Testing**: Tested the model with sample and custom texts
6. **✅ Export**: Downloaded trained models for deployment

### Next Steps:

- **Deploy your model**: Use the downloaded models in your applications
- **Improve performance**: Try different hyperparameters or data augmentation
- **Scale up**: Use larger datasets for better generalization
- **Monitor performance**: Test on real-world data and retrain as needed

### Model Files Included:

- `best_model/`: Best performing model during training
- `final_model/`: Final model after all epochs
- `config.json`: Training configuration
- `training_history.pkl`: Training metrics
- `*.png`: Visualization plots

🎊 **Congratulations on successfully training your BERT model!** 🎊
